# LangChain L2 — Level 1 — One tool and the loop written by hand
OpsPilot v0 cannot compute reliably: models guess arithmetic. Instead of asking the model to
multiply, we give it a **tool**, a Python function it can *request*. This section builds the
entire agent mechanism by hand, in about twenty lines, before any abstraction hides it.

```text
User question
   |
   v
Model  --"call calculate('127 * 834')"-->  YOUR CODE runs calculate()  --"106018"-->  Model
   |                                                                                   |
   +---------------------------- "127 x 834 = 106018" <---------------------------------+
```

The single most important fact about tools: **the model never executes anything.** It emits a
structured request (tool name + arguments). Your application decides whether to run it, runs it,
and sends the result back as a `ToolMessage`. That boundary is where all later safety lives.

### Step 1 — Define a tool with `@tool`

The decorator turns a function into a `BaseTool`: the function name becomes the tool name, the
docstring becomes the description the model reads, and the type hints become the argument
schema. All three are sent to the model with every request, so they are part of your prompt.

In [ ]:
from langchain.tools import tool

@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression such as '127 * 834' and return the numeric result."""
    try:
        return str(eval(expression))        # DANGEROUS: fixed properly in section L4
    except Exception as exc:
        return f"error: {exc}"

print("name        :", calculate.name)
print("description :", calculate.description)
print("args schema :", calculate.args)                      # what the model sees
print("direct call :", calculate.invoke({"expression": "127 * 834"}))   # a tool is also callable by us

### Step 2 — Bind the tool and inspect the request the model makes

`bind_tools()` attaches the tool schemas to the model. The reply for a computation question
is an `AIMessage` with **empty content and a `tool_calls` list**: the model is asking, not answering.

In [ ]:
model_with_tools = model.bind_tools([calculate])

ai = model_with_tools.invoke([HumanMessage("What is 127 * 834?")])
print("content    :", repr(text_of(ai)))
print("tool_calls :", ai.tool_calls)          # [{'name': 'calculate', 'args': {'expression': '127 * 834'}, 'id': ...}]

### Step 3 — Execute the request ourselves and send the result back

Invoking a tool with a tool-call dict returns a ready-made `ToolMessage` whose `tool_call_id`
links the result to the request. Append both and call the model again: now it can answer.

In [ ]:
tool_result = calculate.invoke(ai.tool_calls[0])          # -> ToolMessage
print("tool message :", type(tool_result).__name__, "| id", tool_result.tool_call_id, "| content", tool_result.content)

history = [HumanMessage("What is 127 * 834?"), ai, tool_result]
final = model_with_tools.invoke(history)
print("final answer :", text_of(final))

### Step 4 — The agent loop, written by hand

Generalise Step 3: *while the model keeps requesting tools, execute them and call again.* Add a
step limit so a confused model cannot loop forever. This is the whole agent; `create_agent()`
in the next section is this loop with production features attached.

In [ ]:
def run_agent_by_hand(question, tools, max_steps=5):
    """A minimal agent loop: model -> tool requests -> execute -> model ... -> final text."""
    tool_index = {t.name: t for t in tools}
    llm = model.bind_tools(tools)
    messages = [SystemMessage(OPSPILOT_PERSONA), HumanMessage(question)]
    for step in range(1, max_steps + 1):
        ai = llm.invoke(messages)                       # 1. ask the model what to do next
        messages.append(ai)
        if not ai.tool_calls:                           # 2. no request -> this is the final answer
            print(f"  step {step}: final answer")
            return text_of(ai), messages
        for call in ai.tool_calls:                      # 3. otherwise execute every request WE approve of
            print(f"  step {step}: executing {call['name']}({call['args']})")
            messages.append(tool_index[call["name"]].invoke(call))
    return "Stopped: step limit reached.", messages     # 4. the limit belongs to our code, not the model

answer, trajectory = run_agent_by_hand("What is 127 * 834?", [calculate])
print("ANSWER :", answer)
print("roles  :", [m.type for m in trajectory])

### Recap

- **Problem seen:** the model cannot compute, and it cannot run code either.
- **Layer added:** a `@tool`, `bind_tools()`, `ToolMessage`, and a loop with a step limit that we control.
- **Evidence:** the model's reply was a request (`tool_calls`); the number came from our Python.